# **Week 4: Machine Learning in Astronomy**

## **Part 2 — Evaluating Classification Models and Cross-Validation**

High accuracy does not always mean a useful scientific classifier. This notebook uses a larger exoplanet catalog to study data cleaning, rare classes, precision, recall, confusion matrices, cross-validation, learning curves, and multiclass evaluation.

> **Scientific caution:** Here “habitable” is a catalog category constructed from selected physical criteria. It is not a detection of life, and the three features used below do not capture atmosphere, composition, stellar activity, or many other relevant processes.

### Source and license

This workshop notebook is reformatted from a supplied ASTRO 416 notebook. That source says it accompanies Chapter 3 of Viviana Acquaviva's [*Machine Learning for Physics and Astronomy*](https://books.google.com/books?id=SE2zEAAAQBAJ) (Princeton University Press, 2023), identifies **Viviana Acquaviva (2023)** as copyright holder, and says it was adapted from University of Michigan ASTRO 416 by [Lía Corrales](https://sites.lsa.umich.edu/liacorrales/teaching/). It uses the [BSD 3-Clause License](https://opensource.org/license/bsd-3-clause/). The catalog comes from the [Planetary Habitability Laboratory's Habitable Exoplanets Catalog](https://phl.upr.edu/projects/habitable-exoplanets-catalog).

### **Learning goals**

By the end of this notebook, you should be able to:

- explore a table and select physically motivated features;
- keep features and labels aligned while removing missing values and simple outliers;
- explain why accuracy can be misleading for an imbalanced dataset;
- calculate and interpret accuracy, precision, recall, F1 score, and a confusion matrix;
- compare ordinary, shuffled, and stratified k-fold cross-validation;
- use learning curves to recognize high variance (overfitting) and high bias (underfitting);
- compare scaling choices for kNN without data leakage; and
- evaluate a three-class model without hiding rare-class failures inside one average.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from sklearn import metrics, model_selection, tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, make_scorer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

RANDOM_STATE = 42
FEATURES = ['S_MASS', 'P_PERIOD', 'P_DISTANCE']
FEATURE_LABELS = ['Host-star mass (solar masses)', 'Orbital period (days)', 'Orbital distance (AU)']

plt.rcParams['figure.figsize'] = (8, 5)
pd.set_option('display.max_columns', 20)

# **1. Load and explore the catalog**

The catalog is downloaded directly from a shared Google Drive file. The table has many columns, but we will first inspect its shape, target counts, and three features used in Part 1.

In [ ]:
# Import the exoplanet catalog 
url = "https://drive.google.com/uc?export=download&id=1HtgqGy45R78-oSMCWZ885ME150gcxzrT"  # phl_exoplanet_catalog.csv file
catalog = pd.read_csv(url)

print(f'Catalog shape: {catalog.shape[0]} planets × {catalog.shape[1]} columns')
print('\nFirst 20 column names:')
print(catalog.columns[:20].tolist())
print('\nOriginal habitability categories:')
print(catalog['P_HABITABLE'].value_counts().sort_index())

The original target has three levels:

- `0`: not labeled habitable;
- `1`: possibly habitable;
- `2`: probably habitable.

We first combine categories 1 and 2 into one positive class. This is **binary classification**. The three-level target returns near the end.

In [ ]:
working = catalog[FEATURES + ['P_HABITABLE']].copy()
working['HABITABLE_BINARY'] = (working['P_HABITABLE'] > 0).astype(int)

print(working['HABITABLE_BINARY'].value_counts().sort_index().rename(
    index={0: 'Not labeled habitable', 1: 'Labeled habitable'}
))
display(working.head())

# **2. Feature selection and descriptive statistics**

**Feature selection** chooses which measured quantities enter the model. We use:

- stellar mass (`S_MASS`),
- orbital period (`P_PERIOD`), and
- orbital distance (`P_DISTANCE`).

These are plausible but incomplete. Temperature, incident flux, stellar activity, composition, atmosphere, and measurement uncertainty could matter. Omitting a relevant quantity creates a **latent-variable problem**: the model cannot learn information it never receives.

In [ ]:
all_summary = working[FEATURES].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
positive_summary = working.loc[working['HABITABLE_BINARY'] == 1, FEATURES].describe().T[
    ['count', 'mean', 'std', 'min', '50%', 'max']
]

print('All catalog objects:')
display(all_summary)
print('Objects with a positive catalog label:')
display(positive_summary)

### **Hands-on check**

Compare the medians and ranges in the two tables. Which feature seems to separate the positive class most clearly? A visible difference is useful evidence, but it does not prove causation and does not guarantee good predictions.

# **3. Missing values and simple outlier screening**

Astronomical catalogs are rarely complete. If a planet is missing any selected feature, scikit-learn's decision tree and kNN implementations cannot use that row directly. We remove incomplete rows for this lesson and retain their original indices until features and targets have been filtered together.

In a research analysis, alternatives include imputation, uncertainty-aware modeling, or choosing features with better coverage. Dropping rows can introduce selection bias if the missingness is not random.

In [ ]:
print('Missing values in selected features:')
print(working[FEATURES].isna().sum())

complete = working.dropna(subset=FEATURES).copy()
print(f'\nRows before cleaning: {len(working)}')
print(f'Rows after removing missing selected features: {len(complete)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for feature, label, ax in zip(FEATURES, FEATURE_LABELS, axes):
    values = complete.loc[complete[feature] > 0, feature]
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 60)
    ax.hist(values, bins=bins, color='slateblue', alpha=0.75)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(label)
    ax.set_ylabel('Number of planets')
    ax.set_title(feature)

fig.suptitle('Selected feature distributions (logarithmic axes)')
fig.tight_layout()
plt.show()

Long tails are easy to see on logarithmic axes. To preserve the method used by the source notebook, we demonstrate a simple five-standard-deviation (`5σ`) screen below.

This rule is **not universal**. A z-score is most natural for an approximately normal distribution, while these positive astronomical quantities are strongly skewed. In research, outliers should be checked against measurement flags and physical expectations; an extreme planet may be the most interesting object rather than an error.

In [ ]:
z_scores = np.abs(stats.zscore(complete[FEATURES], nan_policy='omit'))
keep_row = (z_scores < 5).all(axis=1)

model_data = complete.loc[keep_row].reset_index(drop=True)
X = model_data[FEATURES]
y = model_data['HABITABLE_BINARY']

print(f'Complete rows before the 5σ screen: {len(complete)}')
print(f'Rows retained after the 5σ screen:  {len(model_data)}')
print(f'Rows flagged by this simple screen: {len(complete) - len(model_data)}')

# **4. Class imbalance: ask how a model can fail**

The positive class is rare. A model that predicts “not habitable” for every planet can therefore achieve high accuracy while finding none of the scientifically interesting positive examples. This is sometimes called the **accuracy paradox**.

In [ ]:
class_counts = y.value_counts().sort_index()
class_fractions = y.value_counts(normalize=True).sort_index()

balance_table = pd.DataFrame({
    'count': class_counts,
    'fraction': class_fractions,
}).rename(index={0: 'Not labeled habitable', 1: 'Labeled habitable'})

display(balance_table)

ax = balance_table['count'].plot.bar(color=['steelblue', 'tomato'], rot=0)
ax.set_yscale('log')
ax.set_ylabel('Number of planets (log scale)')
ax.set_title('The binary target is strongly imbalanced')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    X['P_PERIOD'],
    X['S_MASS'],
    c=y,
    cmap='coolwarm',
    alpha=0.65,
    s=30,
)
ax.set_xscale('log')
ax.set_xlabel('Orbital period (days)')
ax.set_ylabel('Host-star mass (solar masses)')
ax.set_title('Two selected features colored by the binary catalog label')
ax.legend(*scatter.legend_elements(), title='Catalog label')
plt.show()

### **Think before fitting**

- The classes overlap in this two-dimensional view. What information might a third feature add?
- Would kNN be affected by the dense majority class?
- Which failure is more costly for a follow-up survey: missing a promising planet, or spending telescope time on a false positive?

The last question determines whether **recall** or **precision** deserves more emphasis.

# **5. Train a decision tree and compare a baseline**

We make one stratified train/test split for an initial diagnosis. A `DummyClassifier` supplies an honest baseline by always predicting the most frequent class. A useful ML model should be compared with a simple rule, not only with perfection.

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2,
    stratify=y,
)

dt_model = tree.DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)

dt_train_prediction = dt_model.predict(X_train)
dt_test_prediction = dt_model.predict(X_test)
baseline_prediction = baseline_model.predict(X_test)

print('Decision-tree training accuracy:', metrics.accuracy_score(y_train, dt_train_prediction))
print('Decision-tree test accuracy:    ', metrics.accuracy_score(y_test, dt_test_prediction))
print('Majority-baseline test accuracy:', metrics.accuracy_score(y_test, baseline_prediction))

An unrestricted tree may be too large to read. `max_depth=3` below limits only the **drawing**, not the fitted model. If training accuracy is perfect while test performance is much lower, the tree has high variance and is overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
tree.plot_tree(
    dt_model,
    max_depth=3,
    feature_names=FEATURE_LABELS,
    class_names=['Not labeled habitable', 'Labeled habitable'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax,
)
ax.set_title('Top levels of the fitted decision tree')
plt.show()

# **6. Accuracy, precision, recall, and F1**

For the positive class:

- **Accuracy** = correct predictions / all predictions.
- **Precision** = true positives / all predicted positives. High precision means a positive prediction is usually trustworthy.
- **Recall** = true positives / all actual positives. High recall means few positive objects are missed.
- **F1 score** is the harmonic mean of precision and recall; it is high only when both are reasonably high.

Always pass arguments to scikit-learn metrics in this order: `metric(y_true, y_pred)`.

In [ ]:
def binary_metric_row(name, y_true, y_pred):
    return {
        'model': name,
        'accuracy': metrics.accuracy_score(y_true, y_pred),
        'precision': metrics.precision_score(y_true, y_pred, zero_division=0),
        'recall': metrics.recall_score(y_true, y_pred, zero_division=0),
        'f1': metrics.f1_score(y_true, y_pred, zero_division=0),
    }

metric_table = pd.DataFrame([
    binary_metric_row('Decision tree (test)', y_test, dt_test_prediction),
    binary_metric_row('Majority baseline (test)', y_test, baseline_prediction),
])
display(metric_table.set_index('model').round(3))

# **7. Read a confusion matrix**

Rows are true labels and columns are predicted labels. For binary labels ordered `[0, 1]`, the matrix contains:

$$
\begin{bmatrix}
\mathrm{TN} & \mathrm{FP} \\
\mathrm{FN} & \mathrm{TP}
\end{bmatrix}
$$

The off-diagonal entries are the two kinds of mistakes: false positives and false negatives.

In [ ]:
cm = metrics.confusion_matrix(y_test, dt_test_prediction, labels=[0, 1])

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not labeled habitable', 'Labeled habitable'],
).plot(cmap='Blues', colorbar=False)
plt.title('Decision-tree confusion matrix')
plt.show()

In [ ]:
tn, fp, fn, tp = cm.ravel()
precision_by_hand = tp / (tp + fp) if (tp + fp) else 0.0
recall_by_hand = tp / (tp + fn) if (tp + fn) else 0.0

print(f'TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'Precision by hand: {precision_by_hand:.3f}')
print(f'Recall by hand:    {recall_by_hand:.3f}')

# **8. k-fold cross-validation**

A single split can be lucky or unlucky. In **k-fold cross-validation**, the data are divided into `k` folds. The model trains on `k-1` folds and is evaluated on the remaining fold; this repeats until every fold has served as validation data.

We compare three splitters:

- `KFold`: preserves row order, which is risky if the catalog is sorted.
- shuffled `KFold`: randomizes rows before making folds.
- shuffled `StratifiedKFold`: also keeps the rare-class fraction similar across folds.

Stratification is usually the safest of these choices for imbalanced classification. It does not solve imbalance by itself; it makes evaluation folds more comparable.

In [ ]:
splitters = {
    'KFold (ordered)': model_selection.KFold(n_splits=5),
    'KFold (shuffled)': model_selection.KFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    ),
    'StratifiedKFold': model_selection.StratifiedKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    ),
}

fold_balance = []
for splitter_name, splitter in splitters.items():
    for fold_number, (_, validation_index) in enumerate(splitter.split(X, y), start=1):
        validation_y = y.iloc[validation_index]
        fold_balance.append({
            'splitter': splitter_name,
            'fold': fold_number,
            'validation_size': len(validation_y),
            'positive_count': int(validation_y.sum()),
            'positive_fraction': validation_y.mean(),
        })

display(pd.DataFrame(fold_balance).round(3))

Cross-validation must fit a **fresh model** on each training fold. `cross_validate` clones the estimator for us. The custom scorers define a value of zero when precision or recall has a zero denominator, avoiding undefined-score warnings in unusually sparse folds.

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(metrics.precision_score, zero_division=0),
    'recall': make_scorer(metrics.recall_score, zero_division=0),
    'f1': make_scorer(metrics.f1_score, zero_division=0),
}

cv_rows = []
for splitter_name, splitter in splitters.items():
    scores = model_selection.cross_validate(
        tree.DecisionTreeClassifier(random_state=RANDOM_STATE),
        X,
        y,
        cv=splitter,
        scoring=scoring,
    )
    row = {'splitter': splitter_name}
    for metric_name in scoring:
        values = scores[f'test_{metric_name}']
        row[f'{metric_name}_mean'] = values.mean()
        row[f'{metric_name}_std'] = values.std(ddof=1)
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows).set_index('splitter')
display(cv_summary.round(3))

Do not choose a method from the largest accuracy alone. Compare precision and recall and look at their fold-to-fold variation. Large standard deviations mean that the conclusion depends strongly on which objects land in each fold.

# **9. Learning curves: more data or a different model?**

A learning curve repeats cross-validation with progressively larger training subsets.

- **High variance / overfitting:** training score is high, validation score is lower, and a gap remains. More representative data or stronger regularization may help.
- **High bias / underfitting:** both scores are low and close together. More data alone may not help; better features or a more flexible model may be needed.

Because positive objects are the scientific priority here, we plot recall rather than accuracy.

In [ ]:
def plot_learning_curve(estimator, title, X_values, y_values, cv, scoring, ylim=(0, 1.05)):
    train_sizes, train_scores, validation_scores = model_selection.learning_curve(
        estimator,
        X_values,
        y_values,
        cv=cv,
        scoring=scoring,
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=1,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    validation_mean = validation_scores.mean(axis=1)
    validation_std = validation_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(train_sizes, train_mean, 'o-', label='Training recall', color='firebrick')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                    color='firebrick', alpha=0.15)
    ax.plot(train_sizes, validation_mean, 'o-', label='Validation recall', color='seagreen')
    ax.fill_between(train_sizes, validation_mean - validation_std,
                    validation_mean + validation_std, color='seagreen', alpha=0.15)
    ax.set(xlabel='Number of training examples', ylabel='Recall', title=title, ylim=ylim)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()

    return pd.DataFrame({
        'training_examples': train_sizes,
        'training_recall': train_mean,
        'validation_recall': validation_mean,
    })

In [ ]:
recall_scorer = make_scorer(metrics.recall_score, zero_division=0)
stratified_cv = splitters['StratifiedKFold']

dt_curve = plot_learning_curve(
    tree.DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Decision-tree learning curve',
    X,
    y,
    cv=stratified_cv,
    scoring=recall_scorer,
)
display(dt_curve.round(3))

# **10. Scaling choices for kNN**

`StandardScaler` uses the mean and standard deviation. `RobustScaler` uses the median and interquartile range and is less influenced by long tails. A pipeline ensures that the scaler is fitted only on each training subset.

We compare both scalers on the same held-out test set. The comparison is empirical: the preferred method is the one that best serves the scientific metric and remains stable under cross-validation.

In [ ]:
knn_models = {
    'kNN + StandardScaler': make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=3)
    ),
    'kNN + RobustScaler': make_pipeline(
        RobustScaler(), KNeighborsClassifier(n_neighbors=3)
    ),
}

knn_rows = []
for name, model in knn_models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    knn_rows.append(binary_metric_row(name, y_test, prediction))

knn_comparison = pd.DataFrame(knn_rows).set_index('model')
display(knn_comparison.round(3))

In [ ]:
knn_curve = plot_learning_curve(
    make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=3)),
    'Robust-scaled kNN learning curve',
    X,
    y,
    cv=stratified_cv,
    scoring=recall_scorer,
)
display(knn_curve.round(3))

### **Interpret the curves**

- If training and validation recall are both low and close, the model has high bias for the positive class.
- If training recall is much higher than validation recall, it has high variance.
- If the bands are wide, performance changes substantially across folds.

For this problem, better physical features, class-sensitive training, and more positive examples may matter more than simply adding majority-class planets.

# **11. Return to three-class classification**

Binary labels hide the distinction between “possibly” and “probably” habitable. We now preserve the original `P_HABITABLE` values. The two positive subclasses are both rare, so a large overall accuracy can coexist with poor performance on either one.

We use a depth-limited, class-weighted decision tree as a compact demonstration. `class_weight='balanced'` gives mistakes on rare classes more influence during fitting. This can improve rare-class sensitivity, but it does not create new information or guarantee calibrated probabilities.

In [ ]:
X_multi = model_data[FEATURES]
y_multi = model_data['P_HABITABLE'].astype(int)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = model_selection.train_test_split(
    X_multi,
    y_multi,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_multi,
)

multi_model = tree.DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
multi_model.fit(X_train_multi, y_train_multi)
multi_prediction = multi_model.predict(X_test_multi)

print(metrics.classification_report(
    y_test_multi,
    multi_prediction,
    labels=[0, 1, 2],
    target_names=['Not labeled habitable', 'Possibly habitable', 'Probably habitable'],
    zero_division=0,
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test_multi,
    multi_prediction,
    labels=[0, 1, 2],
    display_labels=['Not labeled', 'Possibly', 'Probably'],
    cmap='Purples',
    colorbar=False,
)
plt.title('Three-class decision-tree confusion matrix')
plt.show()

The report shows precision, recall, and F1 **for each class**. The macro average gives every class equal weight; the weighted average is dominated by the numerous class-0 planets. For rare-object searches, always inspect the individual rare classes and the confusion matrix before summarizing performance with one number.

# **Conclusion**

- Explore and clean features and labels together so rows remain aligned.
- Treat automatic outlier rules as flags for investigation, not universal truth.
- Compare against a simple baseline and choose metrics that match the scientific cost of errors.
- Use `y_true` before `y_pred` in metric functions.
- Prefer stratified cross-validation when classes are rare, and keep preprocessing inside a pipeline.
- Learning curves separate “collect more data” from “change the features or model.”
- For multiclass problems, inspect every class; overall accuracy and weighted averages can hide failures on rare objects.